# S&P 500 Intelligence Oracle: Phase 1 & 2
## Data Ingestion & Scalable Storage Setup

### Project Objective
This project implements an end-to-end Big Data pipeline to predict stock price movements by integrating historical market data with financial news sentiment. We follow the **CRISP-DM** process to ensure a top-tier execution of the data engineering and machine learning workflow.

### 1. Data Sources
* **Source 1: Historical Market Data** – Daily S&P 500 OHLCV price data (1927–present).
* **Source 2: FNSPID (Financial News and Stock Price Integration Dataset)** – 15.7M financial news records from NASDAQ, Bloomberg, Reuters, and Benzinga covering 4,775 S&P 500 companies (1999–2023), with ChatGPT-labeled sentiment scores. Available at: https://huggingface.co/datasets/Zihan1004/FNSPID

### 2. Basic Architecture: Data Storage
According to the project requirements, all raw data must be stored in a **scalable storage solution**. This notebook performs the following:
1. **Initialises the Spark Session** as the core processing engine.
2. **Loads SP500 price data** from the local raw CSV.
3. **Streams FNSPID** from Hugging Face and converts it to a Spark DataFrame.
4. **Converts and stores both datasets in Parquet format** with Snappy compression.

In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SP500_Storage_Setup") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .getOrCreate()

RAW_DATA_DIR = "../data/raw/"
PROCESSED_DATA_DIR = "../data/processed/"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# --- Source 1: S&P 500 Historical Market Data ---
print("Ingesting SP500.csv...")
market_df = spark.read.csv(os.path.join(RAW_DATA_DIR, "SP500.csv"), header=True, inferSchema=True)
market_df.write.mode("overwrite").parquet(os.path.join(PROCESSED_DATA_DIR, "market_data_parquet"))
print(f"Stored {market_df.count():,} rows -> market_data_parquet\n")
market_df.show(5)

+----------+-----------+-----------+-----------+-----------+------+
|      Date|      Close|       High|        Low|       Open|Volume|
+----------+-----------+-----------+-----------+-----------+------+
|12/30/1927|17.65999985|17.65999985|17.65999985|17.65999985|     0|
|  1/3/1928|17.76000023|17.76000023|17.76000023|17.76000023|     0|
|  1/4/1928|17.71999931|17.71999931|17.71999931|17.71999931|     0|
|  1/5/1928|17.54999924|17.54999924|17.54999924|17.54999924|     0|
|  1/6/1928|17.65999985|17.65999985|17.65999985|17.65999985|     0|
+----------+-----------+-----------+-----------+-----------+------+
only showing top 5 rows


### Source 2: FNSPID — Financial News Sentiment

FNSPID is loaded from Hugging Face. The dataset contains article headlines but **no pre-labeled sentiment column** — those fields (`Author`, `Article`, summary columns) are all null in the public release. We therefore:

1. Keep only the four useful columns: `Date`, `Article_title`, `Stock_symbol`, `Publisher`
2. Parse the UTC timestamp to a plain date
3. Compute a `Sentiment_Score ∈ [−1, 1]` from `Article_title` using **VADER** (Valence Aware Dictionary and sEntiment Reasoner), which is specifically tuned for short financial/news text
4. Convert via PyArrow for robust Spark type inference

In [2]:
import pandas as pd
from datasets import load_dataset

# Set to None to load the full 15.7M-record dataset (requires ~8 GB RAM).
# Use an integer (e.g. 500_000) for faster iteration during development.
MAX_RECORDS = 500_000

print("Streaming FNSPID from Hugging Face...")
hf_stream = load_dataset("Zihan1004/FNSPID", split="train", streaming=True)

if MAX_RECORDS is not None:
    hf_stream = hf_stream.take(MAX_RECORDS)

fnspid_pdf = pd.DataFrame(hf_stream)
print(f"Loaded {len(fnspid_pdf):,} records")
print("Columns:", fnspid_pdf.columns.tolist())
fnspid_pdf.head(3)

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,2020-06-05 06:30:54 UTC,Stocks That Hit 52-Week Highs On Friday,A,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,None,None,None,None,None,None
1,2020-06-03 06:45:20 UTC,Stocks That Hit 52-Week Highs On Wednesday,A,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,None,None,None,None,None,None
2,2020-05-26 00:30:07 UTC,71 Biggest Movers From Friday,A,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,None,None,None,None,None,None


### Step 3 — Sentiment Scoring & Spark Conversion

In [3]:
import pyarrow as pa
from pyspark.sql import functions as F
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Keep only the columns we need (drops all-null columns)
KEEP_COLS = ["Date", "Article_title", "Stock_symbol", "Publisher"]
fnspid_pdf = fnspid_pdf[[c for c in KEEP_COLS if c in fnspid_pdf.columns]].copy()

# 2. Parse UTC timestamp string → Python date (e.g. "2020-06-05 06:30:54 UTC" → 2020-06-05)
fnspid_pdf["Date"] = pd.to_datetime(fnspid_pdf["Date"], utc=True).dt.date

# 3. Compute VADER sentiment on Article_title (-1 = most negative, +1 = most positive)
print("Computing VADER sentiment scores...")
analyzer = SentimentIntensityAnalyzer()
fnspid_pdf["Sentiment_Score"] = (
    fnspid_pdf["Article_title"]
    .fillna("")
    .apply(lambda text: analyzer.polarity_scores(text)["compound"])
)
print(f"Score range: {fnspid_pdf['Sentiment_Score'].min():.3f} → {fnspid_pdf['Sentiment_Score'].max():.3f}")
print(f"Positive: {(fnspid_pdf['Sentiment_Score'] > 0).mean():.1%}  "
      f"Neutral: {(fnspid_pdf['Sentiment_Score'] == 0).mean():.1%}  "
      f"Negative: {(fnspid_pdf['Sentiment_Score'] < 0).mean():.1%}\n")

# 4. Convert via PyArrow — avoids Spark type-inference errors on object columns
arrow_table = pa.Table.from_pandas(fnspid_pdf, preserve_index=False)
sentiment_df = spark.createDataFrame(arrow_table)

# 5. Cast Date to Spark DateType
sentiment_df = sentiment_df.withColumn("Date", F.col("Date").cast("date"))

print("FNSPID schema:")
sentiment_df.printSchema()
sentiment_df.show(5, truncate=True)

+----------+--------------------+------------+-----------------+---------------+
|      Date|       Article_title|Stock_symbol|        Publisher|Sentiment_Score|
+----------+--------------------+------------+-----------------+---------------+
|2020-06-05|Stocks That Hit 5...|           A|Benzinga Insights|            0.0|
|2020-06-03|Stocks That Hit 5...|           A|Benzinga Insights|            0.0|
|2020-05-26|71 Biggest Movers...|           A|       Lisa Levin|            0.0|
|2020-05-22|46 Stocks Moving ...|           A|       Lisa Levin|            0.0|
|2020-05-22|B of A Securities...|           A|       Vick Meyer|          0.296|
+----------+--------------------+------------+-----------------+---------------+
only showing top 5 rows


### Step 4 — Save to Parquet

In [4]:
output_path = os.path.join(PROCESSED_DATA_DIR, "sentiment_data_parquet")
sentiment_df.write.mode("overwrite").parquet(output_path)

print(f"Stored {sentiment_df.count():,} rows -> sentiment_data_parquet")

print("\nDate range covered:")
sentiment_df.select(F.min("Date").alias("earliest"), F.max("Date").alias("latest")).show()

print("Sentiment_Score statistics:")
sentiment_df.select(
    F.round(F.avg("Sentiment_Score"), 4).alias("mean"),
    F.round(F.stddev("Sentiment_Score"), 4).alias("std"),
    F.round(F.min("Sentiment_Score"), 4).alias("min"),
    F.round(F.max("Sentiment_Score"), 4).alias("max"),
).show()

print("Storage Layer Setup Complete.")

+------+------+-------+------+
|  mean|   std|    min|   max|
+------+------+-------+------+
|0.0641|0.2722|-0.9578|0.9697|
+------+------+-------+------+

Storage Layer Setup Complete.
